# Basic Simulation — 32 nm Line/Space

This tutorial runs an end-to-end EUV lithography simulation with the default settings:
1. **Mask diffraction orders** from the thin-mask model: 60 nm Ta absorber lines on a Ru-capped Mo/Si multilayer (`use_rcwa=True` switches to RCWA)
2. **Aerial image** from those orders with a Hopkins transmission cross-coefficient (NA 0.33, σ 0.8, conventional illumination)
3. **Resist**: the default `aerial_threshold` model cuts the image at a dose-scaled threshold (`resist_model="full_chem"` runs Dill exposure, PEB and Mack development instead)
4. **CD extraction** from the resist profile

No GPU required — runs on CPU in about a second.

In [ ]:
import matplotlib.pyplot as plt

from euvsimulator.pipeline import SimulationConfig, run_simulation

In [ ]:
cfg = SimulationConfig(
    period_nm=64.0,      # 64 nm pitch (wafer scale)
    line_width_nm=32.0,  # 32 nm lines
    dose_mj_cm2=20.0,    # 20 mJ/cm² dose
    na=0.33,             # numerical aperture
    sigma=0.8,           # partial coherence
    grid=256,            # simulation grid
    n_rcwa_orders=21,    # diffraction orders kept (Fourier orders of the thin mask)
)

result = run_simulation(cfg)
print(f"CD = {result.cd_nm:.2f} nm")
print(f"NILS = {result.nils_value:.4f}")
print(f"Open multilayer reflectivity = {result.clear_field_reflectivity:.4f}")
print(f"Absorber reflectivity = {result.absorber_reflectivity:.4f}")
print(f"Pattern-averaged reflectivity = {result.mask_mean_reflectivity:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(result.aerial_image.cpu().numpy(), cmap='hot', aspect='equal')
ax1.set_title('Aerial image [mJ/cm²]')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(result.resist_profile.cpu().numpy(), cmap='gray', aspect='equal')
ax2.set_title(f'Resist profile (1 = developed) — CD = {result.cd_nm:.1f} nm')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

## Interpret the results

- **CD ≈ 27.29 nm** for a 32 nm design line: 20 mJ/cm² is above this model's dose-to-size (about 13 mJ/cm²), so the line prints narrower than drawn.
- **NILS ≈ 4.50** at the printed edge.
- **Aerial image** is the dose delivered to the resist in mJ/cm²: an open frame would read exactly the dose, the spaces peak at about 18 mJ/cm².
- **Resist profile**: 1 (white) = developed, 0 (black) = resist remaining; the black band is the printed line.
- **Reflectivity**: the open Mo/Si multilayer reflects about 65 % at the 6° chief ray, the 60 nm Ta absorber on it about 1.3 % (`absorber_reflectivity`). `mask_mean_reflectivity` is the area-weighted average over absorber and space (about 33 % here).